# 🍊 Predicción de Ventanas de Aplicación de Fitosanitarios — Cítricos · Pisco Elqui
## ML + Deep Learning · Datos Meteorológicos 2022–2024
---
**Autor:** F. Ardiles para M. Valenzuela · **GitHub:** [franciscoantonioardiles-ctrl](https://github.com/franciscoantonioardiles-ctrl)

## Objetivo
Predecir las **ventanas horarias óptimas** para la aplicación de fitosanitarios en cítricos en el Valle del Elqui usando datos meteorológicos de la Estación Pisco Elqui (2022–2024).

## Criterio de Ventana Óptima para Aplicación de Fitosanitarios
| Variable | Rango Óptimo |
|---|---|
| Temperatura del Aire | 15°C – 25°C |
| Humedad Relativa | > 60% |
| Velocidad del Viento | < 2.78 m/s (10 km/h) |
| Radiación Solar | < 50 W/m² |

## Pipeline
| Fase | Descripción |
|---|---|
| **1** | Carga y ETL del Excel meteorológico |
| **2** | Feature engineering + etiquetado |
| **3** | EDA — análisis temporal y distribuciones |
| **4** | Modelo Random Forest |
| **5** | Red Neuronal Dense (Deep Learning) |
| **6** | Evaluación y exportación de predicciones |

---
## ⚙️ FASE 0 — Instalación de Dependencias

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow openpyxl --quiet
print('✅ Librerías instaladas')

---
## 📂 FASE 1 — Carga y ETL del Dataset Meteorológico

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Cargar el archivo Excel ───────────────────────────────────────────────────
# En Google Colab:
# from google.colab import files
# uploaded = files.upload()
# df_raw = pd.read_excel(next(iter(uploaded)), sheet_name='Sheet1', header=None)

# En local o Jupyter:
df_raw = pd.read_excel('ESTACIONPISCOELQUI3 AÑOS.xlsx', sheet_name='Sheet1', header=None)

# ── Detectar fila de inicio de datos ─────────────────────────────────────────
start_row = df_raw[df_raw.iloc[:, 0] == '2022-01-01 00:00:00'].index[0]
columns   = df_raw.iloc[start_row - 1]
df        = df_raw.iloc[start_row:].copy()
df.columns = columns
df = df[pd.to_datetime(df[df.columns[0]], errors='coerce').notna()]
df.rename(columns={df.columns[0]: 'FechaHora'}, inplace=True)
df['FechaHora'] = pd.to_datetime(df['FechaHora'])

print(f'Shape raw: {df.shape}')
print(f'Período: {df.FechaHora.min()} → {df.FechaHora.max()}')
df.head()

In [ ]:
# ── Seleccionar y renombrar variables ─────────────────────────────────────────
df_model = df[[
    'FechaHora',
    '[Prom] °C Temperatura del Aire[1.5m]',
    '[Prom] % Humedad Relativa[1.5m]',
    '[Prom] m/s Velocidad de Viento[2m]',
    '[Prom] W/m² RadiaciÃ³n Solar[2m]',
    '[Prom] °C Punto de RocÃ­o[2m]'
]].copy()

df_model.columns = ['FechaHora','Temp_C','HR_%','Viento_m_s','Radiacion_W_m2','PuntoRocio_C']
df_model['HoraInt'] = df_model['FechaHora'].dt.hour
df_model['Mes']     = df_model['FechaHora'].dt.month
df_model = df_model.dropna()

print(f'Registros después de limpieza: {len(df_model):,}')
print(f'Nulos: {df_model.isnull().sum().sum()}')
df_model.describe().round(2)

---
## 🏷️ FASE 2 — Feature Engineering: Etiquetado de Ventana Óptima

In [ ]:
def evaluar_ventana(row):
    """
    Ventana óptima para aplicación de fitosanitarios en cítricos:
    - Temperatura entre 15°C y 25°C
    - Humedad Relativa > 60%
    - Viento < 2.78 m/s (10 km/h)
    - Radiación Solar < 50 W/m²
    """
    return int(
        15 <= row['Temp_C'] <= 25 and
        row['HR_%'] > 60 and
        row['Viento_m_s'] < 2.78 and
        row['Radiacion_W_m2'] < 50
    )

df_model['VentanaRecomendada'] = df_model.apply(evaluar_ventana, axis=1)

n_ventana = df_model['VentanaRecomendada'].sum()
n_total   = len(df_model)
print(f'Horas con ventana óptima: {n_ventana:,} / {n_total:,} ({n_ventana/n_total*100:.1f}%)')
print(f'Horas fuera de ventana:   {n_total-n_ventana:,} ({(1-n_ventana/n_total)*100:.1f}%)')

---
## 📊 FASE 3 — EDA: Análisis Temporal de Ventanas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por hora
hourly = df_model.groupby('HoraInt')['VentanaRecomendada'].mean() * 100
colors_h = ['#3fb950' if v>30 else '#d29922' if v>10 else '#f85149' for v in hourly.values]
axes[0].bar(hourly.index, hourly.values, color=colors_h, alpha=0.85)
axes[0].set_title('% Horas con Ventana Óptima por Hora del Día')
axes[0].set_xlabel('Hora'); axes[0].set_ylabel('%')
axes[0].set_xticks(range(0, 24, 2))

# Por mes
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
monthly = df_model.groupby('Mes')['VentanaRecomendada'].mean() * 100
axes[1].bar(range(1,13), monthly.values, color='steelblue', alpha=0.85)
axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(meses, rotation=30, fontsize=8)
axes[1].set_title('% Horas con Ventana Óptima por Mes')
for i,v in enumerate(monthly.values):
    axes[1].text(i+1, v+0.3, f'{v:.1f}%', ha='center', fontsize=7)

plt.suptitle('Análisis Temporal de Ventanas de Aplicación · Pisco Elqui', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots: condiciones ventana vs no ventana
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, ['Temp_C','HR_%','Viento_m_s','Radiacion_W_m2']):
    d0 = df_model[df_model['VentanaRecomendada']==0][col].values
    d1 = df_model[df_model['VentanaRecomendada']==1][col].values
    bp = ax.boxplot([d0, d1], patch_artist=True)
    bp['boxes'][0].set_facecolor('#f85149'); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#3fb950'); bp['boxes'][1].set_alpha(0.7)
    ax.set_xticklabels(['No Ventana','Ventana ✓'], fontsize=8)
    ax.set_title(col, fontsize=9)
plt.suptitle('Condiciones: Ventana vs No Ventana', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🌲 FASE 4 — Modelo Random Forest

In [ ]:
FEATURES = ['Temp_C','HR_%','Viento_m_s','Radiacion_W_m2','PuntoRocio_C','HoraInt','Mes']
X = df_model[FEATURES].astype(float)
y = df_model['VentanaRecomendada']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest Report ===')
print(classification_report(y_test, y_pred_rf, target_names=['No Ventana','Ventana ✓']))
print(f'Accuracy: {rf.score(X_test, y_test)*100:.2f}%')

# Feature importance
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fi.plot(kind='barh', figsize=(8,5), color='steelblue', alpha=0.85)
plt.title('Feature Importance — Random Forest')
plt.tight_layout(); plt.show()

---
## 🧠 FASE 5 — Red Neuronal (Deep Learning)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tr_dl, X_te_dl, y_tr_dl, y_te_dl = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(16, activation='relu'),
    Dense(8,  activation='relu'),
    Dense(1,  activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
model.summary()

history = model.fit(X_tr_dl, y_tr_dl,
                    validation_split=0.1, epochs=20, batch_size=32, verbose=1)

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train',      color='#58a6ff')
axes[0].plot(history.history['val_accuracy'], label='Validación', color='#3fb950')
axes[0].set_title('Accuracy por Época'); axes[0].legend()
axes[1].plot(history.history['loss'],     label='Train Loss', color='#d29922')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='#f85149')
axes[1].set_title('Loss por Época'); axes[1].legend()
plt.suptitle('Curvas de Entrenamiento — Red Neuronal', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Evaluación DL
y_pred_prob = model.predict(X_te_dl).flatten()
y_pred_dl   = (y_pred_prob > 0.5).astype(int)

print('=== Deep Learning Report ===')
print(classification_report(y_te_dl, y_pred_dl, target_names=['No Ventana','Ventana ✓']))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (y_p, title) in zip(axes, [(y_pred_rf, 'Random Forest'), (y_pred_dl, 'Deep Learning')]):
    cm = confusion_matrix(y_test if title=='Random Forest' else y_te_dl, y_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Ventana','Ventana'], yticklabels=['No Ventana','Ventana'], ax=ax)
    ax.set_title(f'Matriz de Confusión — {title}')
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout(); plt.show()

---
## 💾 FASE 6 — Exportación de Predicciones

In [ ]:
# Generar predicciones sobre todo el dataset
df_model['Pred_RF'] = rf.predict(X)
df_model['Pred_DL'] = (model.predict(scaler.transform(X)).flatten() > 0.5).astype(int)

df_model.to_csv('predicciones_mandarina.csv', index=False)
print(f'✅ Exportado: predicciones_mandarina.csv ({len(df_model):,} filas)')
print(f'Horas predichas como ventana (RF): {df_model["Pred_RF"].sum():,}')
print(f'Horas predichas como ventana (DL): {df_model["Pred_DL"].sum():,}')

# Descargar en Colab
# from google.colab import files
# files.download('predicciones_mandarina.csv')

---
## ✅ Conclusiones

| # | Hallazgo | Impacto / Recomendación |
|---|---|---|
| 1 | RF 100% accuracy · DL 99.65% | Ambos modelos listos para sistema de alerta automático |
| 2 | Solo 12.6% de horas son ventana óptima | Programar cosecha en madrugada (00:00–07:00h) |
| 3 | Radiación Solar es la feature más importante | Instalar sensor de radiación de alta precisión |
| 4 | Otoño concentra más ventanas (Abr–Jun) | Ajustar calendario de cosecha para esa temporada |
| 5 | Desbalance de clases afecta precisión DL | Aplicar SMOTE o class_weight en versión futura |

---
*F. Ardiles para M. Valenzuela · github.com/franciscoantonioardiles-ctrl*